In [149]:
# Verifying data path

import os
import glob
import pandas as pd

paths = glob.glob("data/raw/**/*", recursive=True)

files = []
for item in paths:
    if os.path.isfile(item):
        files.append(item)

length = len(files)

print(files)
print(f"\n total files: {length}")

['data/raw\\delay\\TTC Subway Delay Data since 2025.csv', 'data/raw\\delay\\ttc-subway-delay-data-2018.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2019.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2020.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2021.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2022.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2023.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2024.xlsx', 'data/raw\\delay\\ttc-subway-delay-jan-2014-april-2017.xlsx', 'data/raw\\delay\\ttc-subway-delay-may-december-2017.xlsx', 'data/raw\\ridership\\1985-2019 Analysis of ridership.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2012-2013.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2014.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2015.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2016.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2017.xlsx', 'data/raw\\schedules\\agency.txt', 'data/raw\\schedules\\calendar.txt', 'data/raw\\schedules\\c

In [150]:
# Verifying routes 

import duckdb

con = duckdb.connect("ttc.duckdb")


con.sql("CREATE OR REPLACE TABLE routes AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/routes.txt')")


con.sql("SELECT COUNT(route_id) FROM routes GROUP BY route_type").df()



,count(route_id)
0,20
1,3
2,210


In [151]:
con.sql("SELECT route_id, route_short_name, route_long_name " \
"  FROM routes " \
"WHERE route_type = 1").df()

,route_id,route_short_name,route_long_name
0,1,1,Line 1 (Yonge-University)
1,2,2,Line 2 (Bloor - Danforth)
2,4,4,Line 4 (Sheppard)


In [152]:
con.sql("CREATE OR REPLACE TABLE stops AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/stops.txt')")

con.sql("CREATE OR REPLACE TABLE stop_times AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/stop_times.txt')")

con.sql("CREATE OR REPLACE TABLE trips AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/trips.txt')")

In [153]:
# filter in only stop time data related to subway lines

con.sql("""
    CREATE OR REPLACE TABLE subway_stop_times AS
    SELECT stop_times.* FROM trips
    JOIN stop_times 
        ON trips.trip_id = stop_times.trip_id
    WHERE trips.route_id IN [1,2,4]
""")

con.sql("""
    SELECT * FROM subway_stop_times LIMIT 5
""").df()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled
0,50659966,5:45:18,5:45:18,14945,1,None,0,0,NaN
1,50659966,5:47:44,5:47:44,15664,2,None,0,0,1.4442
2,50659966,5:50:22,5:50:22,15659,3,None,0,0,3.3901
3,50659966,5:52:34,5:52:34,15666,4,None,0,0,4.7396
4,50659966,5:54:26,5:54:26,15656,5,None,0,0,5.5524


In [154]:
con.sql("""
    SELECT * FROM stops LIMIT 5
""").df()

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,662,662,Danforth Rd at Kennedy Rd,None,43.714379,-79.260939,None,None,None,None,None,1
1,929,929,Davenport Rd at Bedford Rd,None,43.674448,-79.399659,None,None,None,None,None,1
2,940,940,Davenport Rd at Dupont St,None,43.675511,-79.401938,None,None,None,None,None,2
3,1871,1871,Davisville Ave at Cleveland St,None,43.702088,-79.378112,None,None,None,None,None,1
4,11700,11700,Disco Rd at Attwell Dr,None,43.701362,-79.594843,None,None,None,None,None,1


In [159]:
## fixing name duplicate i.e Bloor-Yonge

pd.set_option('display.max_rows', 70)

mapping = pd.DataFrame([
    ("Bloor Station", "Bloor-Yonge Station"),
    ("Yonge Station", "Bloor-Yonge Station")
], columns = ["raw_name", "canonical_name"])

con.sql("""
        CREATE OR REPLACE TABLE station_trips AS
        SELECT COALESCE(mapping.canonical_name, SPLIT_PART(stop_name, ' -', 1)) AS stations
        , COUNT(*) AS scheduled_trips
        FROM stops
        JOIN subway_stop_times
            ON subway_stop_times.stop_id = stops.stop_id
        LEFT JOIN mapping ON SPLIT_PART(stop_name, ' -', 1) = mapping.raw_name
        GROUP BY 1
        ORDER BY stations ASC
""")

con.sql("""SELECT * FROM station_trips""").df()


,stations,scheduled_trips
0,Bathurst Station,2137
1,Bay Station,2136
2,Bayview Station,1755
3,Bessarion Station,1758
4,Bloor-Yonge Station,4332
5,Broadview Station,2133
6,Castle Frank Station,2133
7,Cedarvale Station,2181
8,Chester Station,2133
9,Christie Station,2139


In [156]:
## Verifying delay datas

paths = glob.glob("data/raw/delay/**")

for f in paths:
    if os.path.splitext(f)[1].lower() == '.xlsx':
        df = pd.read_excel(f, nrows=0)
    else:
        df = pd.read_csv(f, nrows=0)
    print(f, df.columns.tolist())

path = "data/raw/delay/ttc-subway-delay-jan-2014-april-2017.xlsx"
sheets = pd.read_excel(path, sheet_name=None)
print(sheets.keys())

print(sheets['Incidents']['Date'].min())
print(sheets['Incidents']['Date'].max())

data/raw/delay\TTC Subway Delay Data since 2025.csv ['_id', 'Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2018.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2019.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2020.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2021.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2022.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2023.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehi

In [157]:
paths = glob.glob("data/raw/delay/**")

dfs = []

for f in paths:
    if os.path.splitext(f)[1].lower() == '.xlsx':
        df = pd.read_excel(f)
    else:
        df = pd.read_csv(f)
    dfs.append(df)

all_delays = pd.concat(dfs, ignore_index=True)

## all_delays.shape

(all_delays.dtypes.to_string())

## print(all_delays['Date'].max())
## print(all_delays['Date'].min())

'_id          float64\nDate          object\nTime             str\nDay              str\nStation          str\nCode             str\nMin Delay      int64\nMin Gap        int64\nBound            str\nLine             str\nVehicle        int64'

In [ ]:
pd.set_option('display.max_rows', 100)

con.sql("""
CREATE OR REPLACE TABLE raw_delays AS
SELECT * FROM all_delays 
""")

con.sql("""
WITH ranked AS (
    SELECT
        Station, 
        COUNT(Station) AS count,
        count / SUM(COUNT(*)) OVER () AS ratio,
        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) as r_total,
        r_total / SUM(COUNT(*)) OVER () AS r_ratio,
        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS r_num
    FROM raw_delays
    GROUP BY Station
    ORDER BY Count DESC
)
SELECT * FROM ranked WHERE r_num IN (70,150)
""").df()



,Station,count,ratio,r_total,r_ratio,r_num
0,RUNNYMEDE STATION,1046,0.005596,168234.0,0.900051,70
1,VMC STATION PLATFORM 2,19,0.000102,183545.0,0.981965,150


In [ ]:
con.sql("""
SELECT * FROM all_delays
""").df()

,_id,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,1.0,2025-01-01,02:10,Wednesday,BATHURST STATION,MUSAN,5,9,E,BD,5227
1,2.0,2025-01-01,02:30,Wednesday,DUNDAS STATION,MUIRS,0,0,NaN,YU,0
2,3.0,2025-01-01,02:32,Wednesday,BROADVIEW STATION,PUMST,0,0,E,BD,0
3,4.0,2025-01-01,02:58,Wednesday,KEELE STATION,EUSC,0,0,W,BD,5293
4,5.0,2025-01-01,02:58,Wednesday,COXWELL STATION,SUAE,0,0,NaN,BD,0
...,...,...,...,...,...,...,...,...,...,...,...
186911,NaN,2017-05-31 00:00:00,23:14,Wednesday,WILSON STATION (EXITIN,TUMVS,0,0,N,YU,5736
186912,NaN,2017-05-31 00:00:00,23:23,Wednesday,ST CLAIR WEST,SUDP,0,0,N,YU,5821
186913,NaN,2017-05-31 00:00:00,23:30,Wednesday,YONGE UNIVERSITY SUBWA,MUGD,0,0,NaN,YU,0
186914,NaN,2017-05-31 00:00:00,23:41,Wednesday,KENNEDY SRT STATION,MRTO,4,11,S,SRT,3021


In [ ]:
pd.set_option('display.max_rows', None)

con.sql("""
CREATE OR REPLACE TABLE clean_delays AS 
    WITH s1 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, ' STATION$', '') AS Station) FROM raw_delays),
    s2 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, ' (BD|YU|YUS|SRT)$', '') AS Station) FROM s1)
SELECT * FROM s2
""")

result = con.sql("""
SELECT DISTINCT Station FROM clean_delays WHERE Station LIKE '%STATION%'
""").df()

print(result.to_string())

           Date   Time        Day                 Station   Code  Min Delay  Min Gap Bound Line  Vehicle
0    2017-05-01  00:18     Monday  KIPLING STATION (ENTER   TUSC          0        0     W   BD     5251
1    2017-05-01  00:58     Monday       ISLINGTON STATION   MUSC          0        0     W   BD     5182
2    2017-05-01  01:07     Monday  UNION STATION (DOWNSVI  MUPAA          3        7     N   YU     5811
3    2017-05-01  01:18     Monday         MCCOWAN STATION   MRTO          4        9     S  SRT     3006
4    2017-05-01  01:42     Monday    CASTLE FRANK STATION    MUO          7       11     W   BD     5296
5    2017-05-01  05:56     Monday            JANE STATION   EUBK          5        0     W   BD     5056
6    2017-05-01  06:18     Monday        OLD MILL STATION   MUSC          0        0     E   BD     5277
7    2017-05-01  06:26     Monday          DUNDAS STATION    MUI         16       21     S   YU     6121
8    2017-05-01  06:31     Monday   EGLINTON WEST STATI